# Host Qwen3-0.6B on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dinhqhuy7/notebook-model-host/blob/main/notebooks/colab/colab_qwen3_0_6b.ipynb)

Runs [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) with vLLM and exposes it to the Internet
as a key-protected, OpenAI-compatible API. The notebook only hosts the model: prompts, parameters
and answers stay on the caller's side, and the server records no content.

```
caller ──HTTPS──▶ Cloudflare Tunnel ──▶ proxy (checks key) ──▶ vLLM ──▶ GPU
```

**GPU** (weights 1.4 GiB; other GPUs are checked in step 1):

| GPU | Runs? | Memory left for context |
|---|---|---|
| T4 | ✓ | ~10 GiB |
| L4 | ✓ | ~17 GiB |
| A100 40 GB | ✓ | ~33 GiB |
| A100 80 GB | ✓ | ~69 GiB |
| H100 80 GB | ✓ | ~68 GiB |

**How to run:**

1. **Runtime → Change runtime type:** the notebook opens with T4; pick a stronger GPU if you have one.
2. (Recommended) 🔑 **Secrets** in the left sidebar: add `LLM_API_KEY`, enable Notebook access.
3. **Runtime → Run all.** Keep the tab open while in use.

Step 8 prints the URL and API key. How to call the API:
[docs/api.md](https://github.com/dinhqhuy7/notebook-model-host/blob/main/docs/api.md).

## 1. Check the GPU and Internet

The notebook downloads vLLM, the model and cloudflared, so it needs Internet. vLLM 0.33.0 needs NVIDIA driver ≥ 580 (CUDA 13) and GPU compute capability ≥ 7.0. The
notebook picks `bfloat16` (Ampere and newer) or `float16` (T4), and splits the model across all GPUs
when there are several.

In [ ]:
import socket, subprocess

for host in ("pypi.org", "huggingface.co", "github.com"):
    try:
        socket.getaddrinfo(host, 443)
    except OSError:
        raise RuntimeError(f"No Internet (cannot resolve {host}): check the runtime's network, then Runtime → Disconnect and delete runtime and run again") from None

out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
).stdout.strip()
rows = [[s.strip() for s in line.split(",")] for line in out.splitlines()]
GPU_NAME, DRIVER = rows[0][0], rows[0][2]
GPU_MIB = min(int(r[1]) for r in rows)
CAP = min(float(r[3]) for r in rows)
TP = len(rows) if len(rows) in (1, 2, 4, 8) else 1
DTYPE = "bfloat16" if CAP >= 8.0 else "float16"
assert int(DRIVER.split(".")[0]) >= 580, f"Driver {DRIVER} is older than 580: try another GPU type"
assert CAP >= 7.0, f"vLLM does not support {GPU_NAME}: Runtime → Change runtime type → pick a GPU with enough memory (table at the top)"
assert GPU_MIB * TP >= 5576, f"{GPU_MIB * TP} MiB is not enough: Runtime → Change runtime type → pick a GPU with enough memory (table at the top)"
print(f"{len(rows)} × {GPU_NAME}, {GPU_MIB} MiB, driver {DRIVER}, compute {CAP} → {DTYPE}, {TP} GPU")

## 2. Settings

- `MAX_MODEL_LEN`: max tokens per request. `auto` = the longest that fits on the GPU.
- `GPU_MEMORY`: fraction of GPU memory for vLLM; lower to 0.85 on out-of-memory errors.
- `ENFORCE_EAGER`: disables CUDA graphs; loads faster with less memory but generates slower.

The API key comes from the `LLM_API_KEY` secret; without it a random key is generated on every run.
Clear outputs (Edit → Clear all outputs) before sharing the notebook: the key is in them.

In [ ]:
import os, secrets

MODEL_ID = "Qwen/Qwen3-0.6B"
SERVED_NAME = "Qwen3-0.6B"
MODEL_DIR = "/content/models/Qwen3-0.6B"
WORK = "/content/llm_host"
PORT, PROXY_PORT = 8000, 8001

MAX_MODEL_LEN = "auto"  #@param {type:"string"}
GPU_MEMORY = 0.9  #@param {type:"number"}
ENFORCE_EAGER = False  #@param {type:"boolean"}

try:
    from google.colab import userdata
    API_KEY = userdata.get("LLM_API_KEY")
except Exception:
    API_KEY = None
print("API key:", "from the LLM_API_KEY secret" if API_KEY else "randomly generated")
API_KEY = API_KEY or secrets.token_urlsafe(32)
os.makedirs(WORK, exist_ok=True)

## 3. Install vLLM

Takes a few minutes. pip then prints red `ERROR: pip's dependency resolver...` lines about
preinstalled packages (google-colab, gradio...): this is expected and harmless, since vLLM runs in its
own process. The install succeeded if the last line shows the vLLM version.

In [ ]:
!pip install -q "vllm==0.33.0"
!vllm --version

## 4. Download the model (~2 GB)

In [ ]:
import shutil
from huggingface_hub import snapshot_download

os.makedirs(MODEL_DIR, exist_ok=True)
free_gb = shutil.disk_usage(MODEL_DIR).free / 1e9
assert free_gb > 2 * 1.1, f"Only {free_gb:.0f} GB of disk left, need ~2 GB"
snapshot_download(MODEL_ID, local_dir=MODEL_DIR)

## 5. Start the server

The server adds no system prompt, sets no sampling parameters (requests that omit sampling parameters get the model's `generation_config.json`), and logs no prompts
or answers. The flags only enable features that requests can use:

- `--reasoning-parser qwen3`: thinking is returned in the `reasoning` field, separate from `content`.
- `--enable-auto-tool-choice --tool-call-parser hermes`: function calls are returned in `tool_calls`.

The server only listens on `127.0.0.1`. The first start takes a few minutes.

In [ ]:
import json, time, urllib.request

LOG = f"{WORK}/server.log"
cmd = [
    "vllm", "serve", MODEL_DIR, "--served-model-name", SERVED_NAME,
    "--host", "127.0.0.1", "--port", str(PORT), "--api-key", API_KEY,
    "--dtype", DTYPE, "--tensor-parallel-size", str(TP),
    "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", str(GPU_MEMORY),
    "--enable-prefix-caching", "--disable-uvicorn-access-log",
    "--reasoning-parser", "qwen3", "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
]
if ENFORCE_EAGER:
    cmd.append("--enforce-eager")
server = subprocess.Popen(cmd, stdout=open(LOG, "ab"), stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, start_new_session=True)


def healthy() -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            return r.status == 200
    except OSError:
        return False


def call(base, path, body=None):
    data = json.dumps(body).encode() if body is not None else None
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
    with urllib.request.urlopen(urllib.request.Request(base + path, data, headers), timeout=300) as r:
        return json.loads(r.read())


end = time.time() + 30 * 60
while time.time() < end and server.poll() is None and not healthy():
    time.sleep(5)
if not healthy():
    print(open(LOG, errors="replace").read()[-4000:])
    raise RuntimeError("The server did not start; see the log above")

LOCAL = f"http://127.0.0.1:{PORT}"
CONTEXT = call(LOCAL, "/v1/models")["data"][0]["max_model_len"]
print(f"Server ready, max context {CONTEXT} tokens")

## 6. Measure speed

One request with no content generates 256 tokens (`ignore_eos`) to measure speed. Nothing is printed
from the answer.

In [ ]:
started = time.time()
reply = call(LOCAL, "/v1/chat/completions", {
    "model": SERVED_NAME, "messages": [{"role": "user", "content": "."}],
    "max_tokens": 256, "ignore_eos": True, "chat_template_kwargs": {"enable_thinking": False},
})
TOKENS_PER_SECOND = reply["usage"]["completion_tokens"] / (time.time() - started)
print(f"~{TOKENS_PER_SECOND:.0f} tokens/second")

## 7. Key-checking proxy

vLLM's `--api-key` does not protect every path (`/invocations`, `/sleep`... stay open). The proxy
is the only way in: it checks the key on every path and only lets the model APIs through. Source and
tests: [proxy/proxy.py](https://github.com/dinhqhuy7/notebook-model-host/blob/main/proxy/proxy.py).

In [ ]:
import sys, urllib.error

PROXY_FILE = f"{WORK}/proxy.py"
with open(PROXY_FILE, "w") as f:
    f.write(r'''"""The only entry point from the Internet, in front of vLLM.

vLLM's `--api-key` only protects /v1, /v2, /inference and /cohere; other paths such as
/invocations (same text generation as /v1) or /sleep stay open. This proxy:
- checks the key on every path: `Authorization: Bearer <key>` (OpenAI) or `x-api-key: <key>` (Anthropic);
- lets through only the model APIs, everything else gets 404;
- logs nothing and forwards responses chunk by chunk (streaming still works).

    python proxy.py <listen port> <vLLM port> <api key>
"""

import secrets
import sys

from aiohttp import ClientSession, ClientTimeout, web

ALLOWED = {
    ("GET", "/v1/models"),
    ("POST", "/v1/chat/completions"),
    ("POST", "/v1/completions"),
    ("POST", "/v1/responses"),
    ("POST", "/v1/messages"),
    ("POST", "/v1/messages/count_tokens"),
    ("POST", "/tokenize"),
    ("POST", "/detokenize"),
}
# Headers to forward; all others (Host, Cookie...) are dropped.
FORWARD_UP = ("Content-Type", "Accept", "anthropic-version", "anthropic-beta")
FORWARD_DOWN = ("Content-Type", "Cache-Control")
SESSION = web.AppKey("session", ClientSession)


def given_key(request: web.Request) -> str:
    auth = request.headers.get("Authorization", "")
    if auth[:7].lower() == "bearer ":
        return auth[7:].strip()
    return request.headers.get("x-api-key", "").strip()


def make_app(upstream: str, key: str) -> web.Application:
    async def handle(request: web.Request) -> web.StreamResponse:
        if (request.method, request.path) not in ALLOWED:
            return web.json_response({"error": "Not found"}, status=404)
        if not secrets.compare_digest(given_key(request).encode(), key.encode()):
            return web.json_response({"error": "Unauthorized"}, status=401)
        headers = {h: request.headers[h] for h in FORWARD_UP if h in request.headers}
        headers["Authorization"] = f"Bearer {key}"
        try:
            upstream_reply = await request.app[SESSION].request(
                request.method, upstream + request.rel_url.path_qs,
                data=await request.read(), headers=headers,
            )
        except Exception as error:  # vLLM is down or restarting
            return web.json_response({"error": f"upstream: {error!r}"}, status=502)
        async with upstream_reply:
            reply = web.StreamResponse(
                status=upstream_reply.status,
                headers={h: upstream_reply.headers[h] for h in FORWARD_DOWN if h in upstream_reply.headers},
            )
            await reply.prepare(request)
            try:
                async for chunk in upstream_reply.content.iter_any():
                    await reply.write(chunk)
                await reply.write_eof()
            except Exception:  # client disconnected midway
                pass
            return reply

    async def start(app: web.Application) -> None:
        app[SESSION] = ClientSession(timeout=ClientTimeout(total=None, sock_connect=10, sock_read=900))

    async def stop(app: web.Application) -> None:
        await app[SESSION].close()

    app = web.Application(client_max_size=32 * 2**20)
    app.router.add_route("*", "/{tail:.*}", handle)
    app.on_startup.append(start)
    app.on_cleanup.append(stop)
    return app


if __name__ == "__main__":
    listen, vllm_port, api_key = int(sys.argv[1]), sys.argv[2], sys.argv[3]
    web.run_app(make_app(f"http://127.0.0.1:{vllm_port}", api_key),
                host="127.0.0.1", port=listen, print=None, access_log=None)
''')
proxy = subprocess.Popen([sys.executable, PROXY_FILE, str(PROXY_PORT), str(PORT), API_KEY],
                         stdout=open(f"{WORK}/proxy.log", "ab"), stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, start_new_session=True)
PROXY = f"http://127.0.0.1:{PROXY_PORT}"
for _ in range(30):
    try:
        call(PROXY, "/v1/models")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(f"The proxy did not start; see {WORK}/proxy.log")

for path, expected in [("/v1/chat/completions", 401), ("/invocations", 404)]:  # POST without key
    try:
        urllib.request.urlopen(urllib.request.Request(PROXY + path, b"{}"))
        raise AssertionError(f"{path} is not blocked")
    except urllib.error.HTTPError as error:
        assert error.code == expected, (path, error.code)
print("Proxy running: no key → 401, admin APIs → 404")

## 8. Expose to the Internet

Cloudflare quick tunnel: no account needed, the URL changes on every run. Requests without
streaming are cut after 100 seconds (error 524); with `stream: true` the answer arrives as it is
generated and is not cut (tested: 124 seconds, 9000 tokens).

In [ ]:
import re

CLOUDFLARED = f"{WORK}/cloudflared"
subprocess.run(["wget", "-q", "-O", CLOUDFLARED, "https://github.com/cloudflare/cloudflared/releases/"
                "download/2026.10.7/cloudflared-linux-amd64"], check=True)
os.chmod(CLOUDFLARED, 0o755)
TUNNEL_LOG = f"{WORK}/tunnel.log"
tunnel = subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", PROXY],
                          stdout=open(TUNNEL_LOG, "ab"), stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, start_new_session=True)
PUBLIC = None
for _ in range(60):
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
    if found:
        PUBLIC = found[-1]
        break
    time.sleep(2)
assert PUBLIC, f"Could not get the tunnel URL; see {TUNNEL_LOG}"
for _ in range(30):  # DNS for a new URL takes a few seconds
    try:
        call(PUBLIC, "/v1/models")
        break
    except OSError:
        time.sleep(3)
else:
    raise RuntimeError(f"Could not reach the server through the tunnel; see {TUNNEL_LOG}")

print(f'''
Base URL : {PUBLIC}/v1
API key  : {API_KEY}
Model    : {SERVED_NAME}
Context  : {CONTEXT} tokens
Speed    : ~{TOKENS_PER_SECOND:.0f} tokens/second; without streaming keep max_tokens below ~{int(TOKENS_PER_SECOND * 90)} (100-second limit)

curl {PUBLIC}/v1/chat/completions \\
  -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" \\
  -d '{{"model": "{SERVED_NAME}", "messages": [{{"role": "user", "content": "Hello"}}], "max_tokens": 512}}'
''')

## 9. Monitor

Prints status and throughput stats every minute. Stop this cell (■) before running Shut down: the
notebook runs one cell at a time. Stopping it does not stop the server.

In [ ]:
PARTS = {"vLLM": server, "proxy": proxy, "tunnel": tunnel}
try:
    while True:
        dead = [name for name, process in PARTS.items() if process.poll() is not None]
        stats = [l for l in open(LOG, errors="replace").read().splitlines() if "throughput" in l]
        state = f"ERROR: {', '.join(dead)} stopped" if dead else ("OK" if healthy() else "vLLM /health not OK")
        print(time.strftime("%H:%M:%S"), state, "|", stats[-1][max(stats[-1].find("Avg prompt"), 0):] if stats else "")
        if dead:
            break
        time.sleep(60)
except KeyboardInterrupt:
    print("Monitor stopped; the server is still running")

## 10. Shut down

Stop the Monitor cell first, then run the cell below to stop the tunnel, proxy and vLLM. Then Runtime → Disconnect and delete runtime to release the GPU.

In [ ]:
import signal

for process in (globals().get(n) for n in ("tunnel", "proxy", "server")):
    if process is not None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
print("Tunnel, proxy and vLLM stopped")